In [2]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..","DTENV"))
Pkg.instantiate()
include("../scripts/TesselationCore.jl")
if size(LOAD_PATH,1) < 4
    push!(LOAD_PATH, joinpath(@__DIR__,"..","scripts"))
end


  Activating project at `c:\Users\Ivan\Desktop\Stuff4School\Thesis\CleanDTFE\DTENV`


4-element Vector{String}:
 "@"
 "@v#.#"
 "@stdlib"
 "c:\\Users\\Ivan\\Desktop\\Stuff4School\\Thesis\\CleanDTFE\\notebooks\\..\\scripts"

In [3]:
using TetGen
using StaticArrays
using GLMakie
using JLD
using BenchmarkTools
using LinearAlgebra
using Plots

import .TesselationCore


In [ ]:
struct BVHTree
    depth::Int
    rightChild::Union{Nothing,BVHTree}
    leftChild::Union{Nothing,BVHTree} #nothing if leaf, tree if node
    data::Vector{Int}  # simplex index
end


struct BVH
    tree::BVHTree
    bbox::Matrix{Float64}   # global bounding box saved only once
end


function generateBVHTree(boxes,depth::Int,limBox::Matrix)
    #read whiteboard code

    indices = 1:size(boxes,3)

    return generateBVHTree(boxes,depth,limBox,indices)
    
end


function generateBVHTree(boxes,depth::Int,limBox::Matrix, indices)
    #at each depth step, gets dimension (x,y,z one after another), finds halway point, creates left and right
    
    if depth == 0
        return BVHTree(0,nothing,nothing,indices)
    end
    
    ax = depth%3 + 1

    mins = boxes[ax,1,:]
    maxs = boxes[ax,2,:]
    
    line = (limBox[ax,2]-limBox[ax,1])/2

    ids = collect(1:size(boxes,3))


    
    leftIDs = ids[mins .≤ line]
    rightIDs = ids[maxs .≥ line]

    leftBox = copy(limBox)
    leftBox[ax,2] = line
    
    rightBox = copy(limBox)
    rightBox[ax,1] = line

    return BVHTree(depth,
    generateBVHTree(boxes[:,:,leftIDs],depth-1,leftBox,indices[leftIDs]),
    generateBVHTree(boxes[:,:,rightIDs],depth-1,rightBox,indices[rightIDs]),
    indices)

end

function BVH(data::Vector,depth::Int)
    boxes = stack([cornerSimplexMatr(simplex) for simplex in data])

    minima = (minimum(boxes[1,1,:]),minimum(boxes[2,1,:]),minimum(boxes[3,1,:]))
    maxima = (maximum(boxes[1,2,:]),maximum(boxes[2,2,:]),maximum(boxes[3,2,:]))

    box = stack([minima,maxima])


    tree = BVHTree(boxes,depth,box)
    

    # generate bounding box; simplify data to just corners
    # call BVHtree to get the tree -> create BVH object
    return BVH(tree,box)
end


function BVH(data::Vector,depth::Int,box::Matrix)
    boxes = stack([cornerSimplexMatr(simplex) for simplex in data])

    tree = BVHTree(boxes,depth,box)
    return BVH(tree,box)
end


function cornerSimplexMatr(simplex)
    return hcat(minimum(simplex,dims=1)',maximum(simplex,dims=1)') 
end


cornerSimplexMatr (generic function with 1 method)

In [5]:
points3d = [TesselationCore.point3(@SVector rand(3)) for _ in 1:10]
coords, tets = TesselationCore.tesselate(points3d)
simplices = Vector([coords[:,tets[i,:]]' for i in 1:size(tets,1)])
simplex = simplices[1]


4×3 adjoint(::Matrix{Float64}) with eltype Float64:
 0.244961  0.619506  0.169562
 0.100209  0.887922  0.909653
 0.675962  0.824573  0.576
 0.788158  0.992794  0.080996

In [84]:

BVH(simplices,1).tree

BVHTree(1, BVHTree(0, nothing, nothing, [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]), BVHTree(0, nothing, nothing, Int64[]), [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20])

In [85]:
BVH(simplices,2).tree

BVHTree(2, BVHTree(1, BVHTree(0, nothing, nothing, [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]), BVHTree(0, nothing, nothing, Int64[]), [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]), BVHTree(1, BVHTree(0, nothing, nothing, Int64[]), BVHTree(0, nothing, nothing, Int64[]), Int64[]), [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20])

In [87]:
@btime collect(1:size(boxes,3))
@btime (1:size(boxes,3))

  123.542 ns (3 allocations: 256 bytes)
  54.518 ns (1 allocation: 32 bytes)


1:20

In [89]:
bbox

3×2 Matrix{Float64}:
 0.0623614  0.967189
 0.0415242  0.992794
 0.080996   0.976973

In [99]:
bbox[ax,1:1]

1-element Vector{Float64}:
 0.06236141048488186

In [ ]:
@btime leftBox = @views [bbox[:, 1] bbox[:, 2]] 
@btime leftBox = copy(bbox)

  803.797 ns (8 allocations: 288 bytes)
  76.082 ns (2 allocations: 128 bytes)


3×2 Matrix{Float64}:
 0.0623614  0.967189
 0.0415242  0.992794
 0.080996   0.976973

In [101]:
bbox

3×2 Matrix{Float64}:
 0.0623614  0.967189
 0.0415242  0.992794
 0.080996   0.976973